# Text to 3D AI Model - Colab + ngrok

This notebook runs the GitHub project on a Colab GPU and exposes the FastAPI frontend/backend through ngrok.

## 1. Clone the GitHub repository

In [ ]:
REPO_URL = "https://github.com/LucyAlex12/Text_to_3d_ai_model.git"

!rm -rf Text_to_3d_ai_model
!git clone {REPO_URL}
%cd Text_to_3d_ai_model

## 2. Install dependencies

If Colab asks you to restart the runtime after installs, restart it, then rerun the cells from the top.

In [ ]:
!pip -q install -r requirements.txt pyngrok huggingface_hub

## 3. Download TripoSR weights

`TripoSR/model.ckpt` is large, so the notebook downloads it from Hugging Face instead of storing it in Git.

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

Path("TripoSR").mkdir(exist_ok=True)

for filename in ["config.yaml", "model.ckpt"]:
    hf_hub_download(
        repo_id="stabilityai/TripoSR",
        filename=filename,
        local_dir="TripoSR"
    )

print("TripoSR checkpoint ready")

## 4. Configure ngrok

Create an ngrok authtoken at https://dashboard.ngrok.com/get-started/your-authtoken.

Recommended: in Colab, open the key icon on the left and add a secret named `NGROK_AUTH_TOKEN`.

Optional: if you reserved an ngrok static domain, add another secret named `NGROK_STATIC_DOMAIN`, for example `your-name.ngrok-free.app`.

In [ ]:
from getpass import getpass
from pyngrok import ngrok

try:
    from google.colab import userdata
    NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
    NGROK_STATIC_DOMAIN = userdata.get("NGROK_STATIC_DOMAIN")
except Exception:
    NGROK_AUTH_TOKEN = None
    NGROK_STATIC_DOMAIN = None

if not NGROK_AUTH_TOKEN:
    NGROK_AUTH_TOKEN = getpass("Paste your ngrok authtoken: ")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

if NGROK_STATIC_DOMAIN:
    print("ngrok configured with static domain:", NGROK_STATIC_DOMAIN)
else:
    print("ngrok configured with a temporary URL")

## 5. Start backend and expose the app

Open the printed ngrok URL in a normal browser tab. Do not use Colab's iframe preview.

In [ ]:
import os
import subprocess
import time
import requests
from pyngrok import ngrok

os.environ["IMAGE_MODEL_KIND"] = "sd15"
os.environ["SDXL_WIDTH"] = "512"
os.environ["SDXL_HEIGHT"] = "512"
os.environ["SDXL_STEPS"] = "20"
os.environ["SDXL_GUIDANCE_SCALE"] = "7.0"
os.environ["TRIPOSR_MAX_MC_RESOLUTION"] = "256"

ngrok.kill()

server = subprocess.Popen(
    ["python", "-m", "uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

print("Starting backend. This can take a few minutes while models load...")

for _ in range(240):
    if server.poll() is not None:
        remaining = server.stdout.read() if server.stdout else ""
        raise RuntimeError("Backend stopped before it was ready. Logs:\n" + remaining[-4000:])
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.ok:
            print("Backend is ready")
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Backend did not become ready. Run the logs cell below.")

if NGROK_STATIC_DOMAIN:
    tunnel = ngrok.connect(8000, "http", domain=NGROK_STATIC_DOMAIN)
else:
    tunnel = ngrok.connect(8000, "http")

public_url = tunnel.public_url

print("\nOpen this public app URL:")
print(public_url)
print("\nKeep this Colab runtime running while using the app.")
print("If ngrok shows a browser warning page, click through once. The app also sends the ngrok-skip-browser-warning header for API/model requests.")

## Optional: view backend logs

Run this only if the app fails or you want to monitor generation logs.

In [ ]:
while True:
    line = server.stdout.readline()
    if not line:
        break
    print(line, end="")